In [25]:
import pandas as pd
import json
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Load dataset (same CSV you used in EDA)
data = pd.read_csv("../data/raw/creditcard.csv")  # update path if needed

# Split data
reference_data = data.sample(frac=0.7, random_state=42)
current_data = data.drop(reference_data.index)

print("Data loaded successfully")

Data loaded successfully


In [26]:
# Strong artificial drift
current_data["Amount"] = current_data["Amount"] * 10

# Apply drift to MANY columns
for col in current_data.columns:
    if col != "Class":
        current_data[col] = current_data[col] * 2

print("Strong drift created ")

Strong drift created 


In [27]:
report = Report(metrics=[DataDriftPreset()])

report.run(
    reference_data=reference_data,
    current_data=current_data
)

with open("../reports/drift_report.json", "w") as f:
    json.dump(report.as_dict(), f)

print("Drift report saved ")

Drift report saved 


In [30]:
with open("../reports/drift_report.json") as f:
    report_dict = json.load(f)

# Directly use summary (NO complex parsing needed)
result = report_dict['metrics'][0]['result']

drifted_columns = result['number_of_drifted_columns']
total_columns = result['number_of_columns']
drift_ratio = drifted_columns / total_columns

print(f"Drifted Columns: {drifted_columns}/{total_columns}")
print(f"Drift Ratio: {drift_ratio:.2f}")

if drift_ratio > 0.5:
    print("!!! ALERT: Data Drift Detected!  Retrain model!")

    # Retrain using updated data
    model = retrain_model(current_data)

else:
    print("No significant drift.")

Drifted Columns: 30/31
Drift Ratio: 0.97
!!! ALERT: Data Drift Detected!  Retrain model!
 Retraining model...
 Retraining completed!
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     25595
           1       0.77      0.61      0.68        38

    accuracy                           1.00     25633
   macro avg       0.88      0.80      0.84     25633
weighted avg       1.00      1.00      1.00     25633



In [33]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def retrain_model(data):
    print(" Retraining model...")

    X = data.drop("Class", axis=1)
    y = data["Class"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # Pipeline = scaling + model
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(" Retraining completed!")
    print(classification_report(y_test, y_pred))

    return model

joblib.dump(model, "../models/latest_model.pkl")
print("Model saved successfully!")



Model saved successfully!
